# M2 action memory launcher
Run on a Colab GPU after the baseline pilot. This notebook creates separate training tasks, collects baseline training trajectories, builds a training-only memory index, and smoke tests M2. Keep the run flags off until each preceding cell succeeds. Results stay in Drive.


In [ ]:
GITHUB_REPO = 'https://github.com/nave-nap/AgenticSDS.git'
REVISION = 'origin/main'  # Pin to a commit SHA after publishing this phase
DRIVE_ROOT = '/content/drive/MyDrive/agent_support'
TRAIN_COUNT = 40
RUN_TRAINING = False
BUILD_MEMORY = False
RUN_M2_SMOKE = False
assert GITHUB_REPO.startswith('https://github.com/')


## 1. Set up the repository, GPU runtime, Drive, and benchmarks


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, urllib.request
os.environ['PYTHONHASHSEED'] = '0'
drive.mount('/content/drive')
repo = Path('/content/agent_support_repo')
if not repo.exists():
    subprocess.run(['git', 'clone', GITHUB_REPO, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo / 'requirements.txt')], check=True)
os.chdir(repo)
root = Path(DRIVE_ROOT)
root.mkdir(parents=True, exist_ok=True)
config_file = Path('/content/alfworld_config.yaml')
if not config_file.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/alfworld/alfworld/master/configs/base_config.yaml', config_file)
data_root = Path.home() / '.cache' / 'alfworld'
if not (data_root / 'json_2.1.1').exists():
    subprocess.run(['alfworld-download'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', commit)


## 2. Create training manifests with development and held-out exclusions


In [ ]:
from agent_support.common import read_json
for benchmark in ('alfworld', 'textcraft'):
    folder = root / 'manifests' / benchmark
    development = folder / 'development.json'
    heldout = folder / 'heldout.json'
    if not development.exists() or not heldout.exists():
        raise RuntimeError(f'Missing evaluation manifests: {folder}')
    training = folder / 'training.json'
    command = [sys.executable, '-m', 'agent_support.training_manifest', benchmark,
               '--count', str(TRAIN_COUNT), '--output', str(training),
               '--exclude-manifest', str(development), '--exclude-manifest', str(heldout)]
    if benchmark == 'alfworld':
        command += ['--config', str(config_file)]
    subprocess.run(command, check=True, cwd=repo)
print('Training manifests ready')


## 3. Collect baseline trajectories from training tasks
Set `RUN_TRAINING=True` in cell 1, rerun cell 1, then run this cell. Each benchmark runs once and saves a separate full run on Drive. Do not rerun this cell unless a run failed.


In [ ]:
def run_visible(command, label):
    from collections import deque
    recent = deque(maxlen=30)
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in process.stdout:
        print(line, end='', flush=True)
        recent.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f'{label} exited with code {code}. Last output:\n' + ''.join(recent))
if RUN_TRAINING:
    for benchmark in ('alfworld', 'textcraft'):
        command = [sys.executable, '-m', 'agent_support.runner',
                   '--config', str(repo / 'configs' / f'{benchmark}_baseline.json'),
                   '--manifest', str(root / 'manifests' / benchmark / 'training.json'),
                   '--output-root', str(root / 'training_runs')]
        print('Collecting training trajectories:', benchmark, flush=True)
        run_visible(command, benchmark)
else:
    print('Training collection is off. Set RUN_TRAINING=True in cell 1 when ready.')


## 4. Build the training-only memory corpus and semantic index
Set `BUILD_MEMORY=True` in cell 1 and rerun cell 1, then run this cell. It requires complete training runs from cell 3.


In [ ]:
from agent_support.common import canonical_digest

def latest_training_run(benchmark):
    expected_manifest = read_json(root / 'manifests' / benchmark / 'training.json')
    expected_hash = canonical_digest(expected_manifest)
    candidates = []
    for directory in (root / 'training_runs').glob(f'{benchmark}_*'):
        metadata_path = directory / 'run.json'
        if not metadata_path.exists():
            continue
        meta = read_json(metadata_path)
        if (meta.get('status') == 'complete' and meta.get('partition') == 'training'
                and meta.get('selected_task_count') == TRAIN_COUNT and meta.get('limit') is None
                and meta.get('selected_manifest_sha256') == expected_hash
                and meta.get('git_commit') == commit):
            candidates.append(directory)
    if not candidates:
        raise RuntimeError(f'No complete current training run for {benchmark}; run cell 3 first')
    return max(candidates, key=lambda path: path.name)

if BUILD_MEMORY:
    for benchmark in ('alfworld', 'textcraft'):
        folder = root / 'manifests' / benchmark
        command = [sys.executable, '-m', 'agent_support.memory',
                   '--run', str(latest_training_run(benchmark)),
                   '--exclude-manifest', str(folder / 'development.json'),
                   '--exclude-manifest', str(folder / 'heldout.json'),
                   '--output-dir', str(root / 'memory' / benchmark)]
        print('Building memory:', benchmark, flush=True)
        run_visible(command, f'{benchmark} memory')
else:
    print('Memory build is off. Set BUILD_MEMORY=True in cell 1 after training runs complete.')


## 5. Smoke test M2 on the development manifests
Set `RUN_M2_SMOKE=True` in cell 1 and rerun cell 1, then run this cell. Each run uses three development tasks. Review its `run.json`, `summary.json`, and `episodes/*.json` before a full M2 pilot.


In [ ]:
if RUN_M2_SMOKE:
    for benchmark in ('alfworld', 'textcraft'):
        memory_dir = root / 'memory' / benchmark
        if not (memory_dir / 'corpus.json').exists() or not (memory_dir / 'index.json').exists():
            raise RuntimeError(f'Missing memory files for {benchmark}; run cell 4 first')
        command = [sys.executable, '-m', 'agent_support.runner',
                   '--config', str(repo / 'configs' / f'{benchmark}_m2.json'),
                   '--manifest', str(root / 'manifests' / benchmark / 'development.json'),
                   '--output-root', str(root / 'results'), '--limit', '3']
        print('M2 smoke test:', benchmark, flush=True)
        run_visible(command, f'{benchmark} M2 smoke')
else:
    print('M2 smoke test is off. Set RUN_M2_SMOKE=True in cell 1 after memory build succeeds.')
